<a href="https://colab.research.google.com/github/Ibrah-N/PNID_Custom_OCR_Training_Detector_Recognizer/blob/main/PNID_OCR_Detection_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [TESTING] Text Detector Model Training

### Repo Clone & Installation

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.9/420.9 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import paddle

paddle.device.get_device()
paddle.set_device("gpu")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [ ]:
!git clone https://github.com/PaddlePaddle/PaddleOCR

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 325904, done.
remote: Counting objects: 100% (1651/1651), done.
remote: Compressing objects: 100% (344/344), done.
remote: Total 325904 (delta 1494), reused 1307 (delta 1307), pack-reused 324253 (from 3)
Receiving objects: 100% (325904/325904), 1.73 GiB | 29.47 MiB/s, done.
Resolving deltas: 100% (258161/258161), done.


In [ ]:
%cd PaddleOCR

!pip install -r requirements.txt

/content/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.2 MB/s eta 0:00:00


### Processing Data

In [14]:
# 下载示例数据集
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/data/ocr_det_dataset_examples.tar
!tar -xf ocr_det_dataset_examples.tar

--2026-05-01 10:05:01--  https://paddle-model-ecology.bj.bcebos.com/paddlex/data/ocr_det_dataset_examples.tar
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22773760 (22M) [application/octet-stream]
Saving to: ‘ocr_det_dataset_examples.tar’

ocr_det_dataset_exa 100%[===================>]  21.72M  5.65MB/s    in 24s     

2026-05-01 10:05:27 (929 KB/s) - ‘ocr_det_dataset_examples.tar’ saved [22773760/22773760]



In [ ]:
# 下载 PP-OCRv5_server_det 预训练模型
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams

--2026-03-07 10:38:05--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105414496 (101M) [application/octet-stream]
Saving to: ‘PP-OCRv5_server_det_pretrained.pdparams’

PP-OCRv5_server_det 100%[===================>] 100.53M  16.8MB/s    in 29s     

2026-03-07 10:38:36 (3.49 MB/s) - ‘PP-OCRv5_server_det_pretrained.pdparams’ saved [105414496/105414496]



### Training Model

In [ ]:
#单卡训练 (默认训练方式)
!python3 tools/train.py -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
    -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
    Train.dataset.data_dir=./ocr_det_dataset_examples \
    Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
    Eval.dataset.data_dir=./ocr_det_dataset_examples \
    Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'


#多卡训练，通过--gpus参数指定卡号
# python3 -m paddle.distributed.launch --gpus '0,1,2,3' tools/train.py \
#     -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
#     -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
#     Train.dataset.data_dir=./ocr_det_dataset_examples \
#     Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
#     Eval.dataset.data_dir=./ocr_det_dataset_examples \
#     Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/03/07 10:38:48] ppocr INFO: Architecture : 
[2026/03/07 10:38:48] ppocr INFO:     Backbone : 
[2026/03/07 10:38:48] ppocr INFO:         det : True
[2026/03/07 10:38:48] ppocr INFO:         name : PPHGNetV2_B4
[2026/03/07 10:38:48] ppocr INFO:     Head : 
[2026/03/07 10:38:48] ppocr INFO:         k : 50
[2026/03/07 10:38:48] ppocr INFO:         mode : large
[2026/03/07 10:38:48] ppocr INFO:         name : PFHeadLocal
[2026/03/07 10:38:48] ppocr INFO:     Neck : 
[2026/03/07 10:38:48] ppocr INFO:         intracl : True
[2026/03/07 10:38:48] ppocr INFO:         name : LKPAN
[2026/03/07 10:38:48] ppocr INFO:         out_chann

# PPOCR_V5_det Training

## Repo Clone & Installation

In [ ]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

In [ ]:
import paddle

paddle.device.get_device()
paddle.set_device("gpu")

In [ ]:
!git clone https://github.com/PaddlePaddle/PaddleOCR

In [ ]:
%cd PaddleOCR

!pip install -r requirements.txt

## Processing Data

In [1]:
from google.colab import drive
drive.mount('/content/Ibrah')

Mounted at /content/Ibrah


### Utils

In [15]:
#====================================#
#   D A T A - L O A D - C L A S S    #
#====================================#

import os
import json
import math
import numpy as np
import cv2


def load_json(json_path):
  """load json from json path"""

  if os.path.exists(json_path) and json_path.lower().endswith(".json"):
    with open(json_path, "r") as f:
      jsons_list = json.load(f)
  else:
    raise ValueError(f"Invalid json path: {json_path}")

  return jsons_list



def read_image(image_path):
  """read image from the image path"""

  if image_path.lower().endswith(".jpg") or image_path.lower().endswith(".png"):
    image = cv2.imread(image_path)
  else:
    print("Invalid image path: {}".format(image_path))
    return None

  return image




def xywhrot_xyxyxyxy(label, percentages=True):
  """
  Converting Label Studio format into Oriented Bounding Box (OBB) format.

  Args:
    label (dict): Dictionary containing annotation information
    percentages (bool): if true it will convert percenatage values into pixels values

  Maths:
    label:
      x, y, width, height, rotation

      if percentages:
        x = x / 100 * original_width
        y = y / 100 * original_height
        w = width / 100 * original_width
        h = height / 100 * original_height

      rotation:
        rotation_ = math.radians(rotation)
        cos, sin = math.cos(rotation_), math.sin(rotation_)

        coords = [
          (x, y),
          (x + w * cos, y + w * sin),
          (x + w * cos - h * sin, y + w * sin + h * cos),
          (x - h * sin, y + h * cos),
        ]

  Returns:
    list of tuple or None: List of tuples containing the coordinates of the object in Yolo OBB
  """

  # unpick values
  original_width = label['original_width']
  original_height = label['original_height']
  x, y = label['x'], label['y']
  w, h = label['width'], label['height']
  rotation = label['rotation']


  if percentages:
    x = x / 100 * original_width
    y = y / 100 * original_height
    w = w / 100 * original_width
    h = h / 100 * original_height

  # rotation -> angles
  rotation = math.radians(rotation)
  cos, sin = math.cos(rotation), math.sin(rotation)

  # xywhr -> xyxyxyxy
  coords = [
      (x, y),
      (x + w * cos, y + w * sin),
      (x + w * cos -h * sin, y + w * sin + h * cos),
      (x - h * sin, y + h * cos)
  ]

  return np.array(coords, dtype=np.int32)





def xyxyxyxy_xywhr(xyxyxyxy, original_width, original_height):
  """
    Convert YOLO Oriented Bounding Box (OBB) format to Label Studio format.

    Args:
        xyxyxyxy (list): List of 8 float values representing the absolute pixel coordinates
                         of the OBB in the format [x1, y1, x2, y2, x3, y3, x4, y4]
        original_width (int): Original width of the image
        original_height (int): Original height of the image


    Return:
        dict: Dictionary containing the converted bounding box
              list: [x, y, width, height, rotation, original_width, original_height]
  """
  # Reshape the coordinates into a 4x2 matrix
  coords = np.array(xyxyxyxy, dtype=np.float64).reshape((4, 2))

  # Calculate the center of the bounding box
  center_x = np.mean(coords[:, 0])
  center_y = np.mean(coords[:, 1])

  # Calculate the width and height of the bounding box
  width = np.linalg.norm(coords[0] - coords[1])
  height = np.linalg.norm(coords[0] - coords[3])

  # Calculate the rotation angle
  dx = coords[1, 0] - coords[0, 0]
  dy = coords[1, 1] - coords[0, 1]
  r = np.degrees(np.arctan2(dy, dx))

  # Find the top-left corner (x, y)
  top_left_x = (
      center_x
      - (width / 2) * np.cos(np.radians(r))
      + (height / 2) * np.sin(np.radians(r))
  )
  top_left_y = (
      center_y
      - (width / 2) * np.sin(np.radians(r))
      - (height / 2) * np.cos(np.radians(r))
  )

  # Normalize the values
  x = (top_left_x / original_width) * 100
  y = (top_left_y / original_height) * 100
  width = (width / original_width) * 100
  height = (height / original_height) * 100

  # Create the dictionary for Label Studio
  return {
      "x": x,
      "y": y,
      "width": width,
      "height": height,
      "rotation": r,
      "original_width": original_width,
      "original_height": original_height,
  }


def put_text_on_polygon_simple(image, polygon, text,
                              font=cv2.FONT_HERSHEY_SIMPLEX,
                              font_scale=0.5,
                              thickness=1,
                              color=(0, 255, 0)):

    polygon = polygon.reshape(-1, 2).astype(int)

    # ---- 1. Get top-left point ----
    # smallest y → top, then smallest x → left
    top_left = sorted(polygon, key=lambda p: (p[1], p[0]))[0]

    x, y = top_left

    # ---- 2. Get text size ----
    (w, h), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    # ---- 3. Adjust position (so text is visible) ----
    y = max(h + 5, y)  # avoid going above image
    x = max(0, x)

    # ---- 4. Optional: background box for visibility ----
    cv2.rectangle(image,
                  (x, y - h - baseline),
                  (x + w, y + baseline),
                  (0, 0, 0))

    # ---- 5. Put horizontal text ----
    cv2.putText(image, text, (x, y),
                font, font_scale, color, thickness, cv2.LINE_AA)

    return image





def per_sample_object(image_path, bboxes, transcriptions):
  """It creates one row object per image

  Args:
    image_path (str): path to image
    bboxes (list): list of bounding boxes
    transcriptions (list): list of transcriptions


  Returns:
    list: list of per sample objects
    Example: [
                images/test.jpg,
                [
                    {
                        "transcription": "hello",
                        "points": [[0, 0], [0, 0], [0, 0], [0, 0]]
                    },
                    {
                        "transcription": "hello",
                        "points": [[0, 0], [0, 0], [0, 0], [0, 0]]
                    },
                    ...
                    ...
                    ...
                ]
            ]
  """

  objects = []
  for bbox, transcription in zip(bboxes, transcriptions):

    # xywhrot --> xyxyxyxy
    obb = xywhrot_xyxyxyxy(bbox)

    # append objects list
    objects.append({
        "transcription": transcription,
        "points": obb.tolist()
    })


  # return
  return [image_path, objects]

### Dataset Visualization Script

In [ ]:
# -- unzip the root data zip --

root_zip_path = "/content/Ibrah/MyDrive/PNID_OCR_V2/Isometric Text Export.zip"
!unzip {root_zip_path.replace(" ", "\ ")} -d /content/annotated_data


# unzip akhila and srvani annotatinos
akhila_annotation_path = "/content/annotated_data/Isometric Text Export/Akhila.zip"
srvani_annotation_path = "/content/annotated_data/Isometric Text Export/SRAVANI.zip"


!unzip {akhila_annotation_path.replace(" ", "\ ")} -d /content/akhila_data
!unzip {srvani_annotation_path.replace(" ", "\ ")} -d /content/srvani_data

In [ ]:
# akhila data paths
'''
json_path = "/content/akhila_data/Akhila/project-3-at-2026-04-29-14-11-639dbd78.json"
root_images_path = "/content/akhila_data/Akhila/ocr_dataset"
save_results_path = "/content/saved_akhila"
'''

# srvani data paths
json_path = "/content/srvani_data/SRAVANI/project-4-at-2026-04-29-13-26-0b18cae9.json"
root_images_path = "/content/srvani_data/SRAVANI/ocr_dataset"
save_results_path = "/content/saved_srvani"


os.makedirs(save_results_path, exist_ok=True)

annotations = load_json(json_path)


# iterate annotations
for anno_idx, annotation in enumerate(annotations):

  # extract per image annotations and metadata
  image_name = annotation['ocr'].split("/")[-1]
  bboxes = annotation['bbox']
  transcriptions = annotation['transcription']


  # read image
  image_path = os.path.join(root_images_path, image_name)
  image = read_image(image_path)


  # loop through each bbox
  for bbox, transcription in zip(bboxes, transcriptions):
    obb = xywhrot_xyxyxyxy(bbox)

    cv2.polylines(image, [obb], True, (0, 255, 0), 2)
    image = put_text_on_polygon_simple(image, obb, transcription,
                        font=cv2.FONT_HERSHEY_SIMPLEX,
                        font_scale=0.5,
                        thickness=1,
                        color=(0, 0, 0))


  save image
  save_path = os.path.join(save_results_path, image_name)
  cv2.imwrite(save_path, image)
  print("Sample NO: {:04d} Processed and Saved Successfully to {}"
        .format(anno_idx, save_path))

### Main

In [5]:
# -- unzip the root data zip --

root_zip_path = "/content/Ibrah/MyDrive/PNID_OCR_V2/Isometric Text Export.zip"
!unzip {root_zip_path.replace(" ", "\ ")} -d /content/annotated_data

<string>:1: SyntaxWarning: invalid escape sequence '\ '


Archive:  /content/Ibrah/MyDrive/PNID_OCR_V2/Isometric Text Export.zip
   creating: /content/annotated_data/Isometric Text Export/
  inflating: /content/annotated_data/Isometric Text Export/Akhila.zip  
  inflating: /content/annotated_data/Isometric Text Export/Bad isometric.zip  
  inflating: /content/annotated_data/Isometric Text Export/Good isometric.zip  
  inflating: /content/annotated_data/Isometric Text Export/SRAVANI.zip  


In [ ]:
# unzip akhila and srvani annotatinos
akhila_annotation_path = "/content/annotated_data/Isometric Text Export/Akhila.zip"
srvani_annotation_path = "/content/annotated_data/Isometric Text Export/SRAVANI.zip"


!unzip {akhila_annotation_path.replace(" ", "\ ")} -d /content/akhila_data
!unzip {srvani_annotation_path.replace(" ", "\ ")} -d /content/srvani_data

In [18]:

if __name__=="__main__":
  CONFIG = {

      # akhila data paths
      'akhila': {
          'json_path' : "/content/akhila_data/Akhila/project-3-at-2026-04-29-14-11-639dbd78.json",
          'images_path' : "/content/akhila_data/Akhila/ocr_dataset"
      },

      # srvani data paths
      'srvani': {
          'json_path' : "/content/srvani_data/SRAVANI/project-4-at-2026-04-29-13-26-0b18cae9.json",
          'images_path' : "/content/srvani_data/SRAVANI/ocr_dataset"
      }
  }


  # det_model dataset structure
  DATASET_STRUCTURE = {
      "images_path" : "det_dataset/images",
      "train_labels_path" : "det_dataset/train.txt",
      "val_labels_path" : "det_dataset/val.txt"
  }

  # create dataset structure
  os.makedirs(DATASET_STRUCTURE['images_path'], exist_ok=True)


  # === parase the CONFIG paths ===
  for annotater_name, annotation_configs in CONFIG.items():

    print("Processing Annotater {} annotated dataset".format(annotater_name))
    # paths to json and images
    json_path = annotation_configs['json_path']
    annotated_images_path = annotation_configs['images_path']


    # read json
    annotations = load_json(json_path)

    # parse annotations for each image
    for anno_idx, annotation in enumerate(annotations):
      print("NO {} : Processing {}".format(anno_idx, annotation.get('ocr').split("/")[-1]))

      # extract iamge_name, bboxes, transcriptions
      image_name = annotation.get('ocr').split("/")[-1]
      bboxes = annotation.get('bbox')
      transcriptions = annotation.get('transcription')

      # image_path for the new dataset
      image_path = os.path.join(DATASET_STRUCTURE['images_path'], image_name)

      # create object
      row_object = per_sample_object(
                        image_path=image_path,
                        bboxes=bboxes,
                        transcriptions=transcriptions
                        )

      # copy image from to new dataset
      copy_image_flag = copy_image(
                        image_path_1=os.path.join(
                                            annotated_images_path,
                                            image_name)
                        image_path_2=image_path
                        )

      if not copy_image_flag:
        print("Failed to copy Image {} to {}".format(image_name, image_path))


      break
    break

['det_dataset/images/1_page1__23b298f3-b16a-4b82-ba68-c30351d13464.jpg', [{'transcription': '25"-PW-7842', 'points': [[448, 1425], [549, 1487], [534, 1512], [433, 1451]]}, {'transcription': 'CONT. ON', 'points': [[144, 1441], [213, 1441], [213, 1460], [144, 1460]]}, {'transcription': 'M02-12"-KA-210601-3SDNB-H1-1.5"-01', 'points': [[142, 1456], [396, 1456], [396, 1475], [142, 1475]]}, {'transcription': 'X222582', 'points': [[146, 1475], [209, 1475], [209, 1488], [146, 1488]]}, {'transcription': 'Z (+)50719', 'points': [[138, 1504], [214, 1506], [214, 1524], [138, 1523]]}, {'transcription': '244', 'points': [[518, 1502], [540, 1516], [533, 1527], [511, 1513]]}, {'transcription': 'LV-123', 'points': [[764, 1612], [838, 1651], [824, 1678], [750, 1639]]}, {'transcription': 'CONT. ON', 'points': [[347, 1624], [418, 1628], [417, 1648], [346, 1644]]}, {'transcription': 'M02-12"-KA-210601-3SDNB-H1-1.5*-14', 'points': [[345, 1643], [588, 1643], [588, 1661], [345, 1661]]}, {'transcription': '301

### Training Model

In [ ]:
# 下载 PP-OCRv5_server_det 预训练模型
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams

In [ ]:
#单卡训练 (默认训练方式)
!python3 tools/train.py -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
    -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
    Train.dataset.data_dir=./ocr_det_dataset_examples \
    Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
    Eval.dataset.data_dir=./ocr_det_dataset_examples \
    Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'


#多卡训练，通过--gpus参数指定卡号
# python3 -m paddle.distributed.launch --gpus '0,1,2,3' tools/train.py \
#     -c configs/det/PP-OCRv5/PP-OCRv5_server_det.yml \
#     -o Global.pretrained_model=./PP-OCRv5_server_det_pretrained.pdparams \
#     Train.dataset.data_dir=./ocr_det_dataset_examples \
#     Train.dataset.label_file_list='[./ocr_det_dataset_examples/train.txt]' \
#     Eval.dataset.data_dir=./ocr_det_dataset_examples \
#     Eval.dataset.label_file_list='[./ocr_det_dataset_examples/val.txt]'

### Testing Model